Test Ollama and DeepSeek Connection

In [21]:
import requests

def test_ollama_connection():
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "deepseek-r1:8b",
        "prompt": "Say hello in one short sentence.",
        "stream": False
    }
    response = requests.post(url, json=payload)
    response.raise_for_status()
    data = response.json()
    print("Model replied:")
    print(data["response"])

test_ollama_connection()

Model replied:
Hello!

Or

Hi there!


Test Ollama and DeepSeek Connection

In [14]:
def load_syllabus(file_path):
    """
    Reads the syllabus text file and returns its content as a single string.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content

syllabus_path = "Sample Curriculum.txt"

syllabus_text = load_syllabus(syllabus_path)

print("File loaded successfully!")
print("Total characters:", len(syllabus_text))
print("\n--- Preview of content ---\n")
print(syllabus_text[:500])  # just show the 500 words as preview

File loaded successfully!
Total characters: 1374

--- Preview of content ---

Sample Topics/Syllabus:

Introduction to Problem Solving: Problem solving steps, algorithms,
pseudo-code, flowcharts, overview of Von-Neumann Architecture 

Overview of the structured program theorem (sequence, selection and
iteration) 

Introduction to C++: Structure of a C++ program, compilation
process (compiler & linker), development environment, simple
program (printing text), Variables and Data Types: Concept of
variables, data types (int, float, char, bool)

Input/output using cin/cout, a


Split the Syllabus into Individual Topics

In [22]:
import re

def split_into_topics(syllabus_text):
    """
    Splits the raw syllabus text into a list of individual topic strings.
    Assumes topics are separated by one or more blank lines.
    """
    
    text = syllabus_text.replace("Sample Topics/Syllabus:", "").strip()

    raw_topics = re.split(r"\n\s*\n", text)

    topics = []
    for topic in raw_topics:
        cleaned = " ".join(topic.split())  
        if cleaned:  
            topics.append(cleaned)

    return topics

topics_list = split_into_topics(syllabus_text)

print(f"Found {len(topics_list)} topics:\n")
for i, topic in enumerate(topics_list, start=1):
    print(f"{i}. {topic}\n")

Found 11 topics:

1. Introduction to Problem Solving: Problem solving steps, algorithms, pseudo-code, flowcharts, overview of Von-Neumann Architecture

2. Overview of the structured program theorem (sequence, selection and iteration)

3. Introduction to C++: Structure of a C++ program, compilation process (compiler & linker), development environment, simple program (printing text), Variables and Data Types: Concept of variables, data types (int, float, char, bool)

4. Input/output using cin/cout, arithmetic expressions, assignment statements, Operators and Expressions: Arithmetic operators, operator precedence, integer division, type casting, introduction to memory concepts

5. Decision Making I: Relational and equality operators, if and if-else statements, nested if

6. Decision Making II & Loops: Logical operators, while loop, structured program development

7. Iteration Control: for, do-while loops, break and continue, introduction to switch statement

8. Functions I: Modular progra

Build the AI Prompt for Question Generation

In [23]:
def build_question_prompt(topic):
    """
    Builds the instruction prompt sent to DeepSeek for generating
    ONE medium-length, 5-mark exam question based on a given topic.
    """
    prompt = f"""You are an experienced computer science exam paper setter.

Topic: {topic}

Write exactly ONE exam question based on this topic, following these strict rules:
- The question must be worth 5 marks.
- The question should be medium length: not a one-liner, and not a multi-part essay question. Aim for 1-3 sentences.
- Do NOT provide the answer or any explanation.
- Do NOT include the words "Answer:", solutions, or hints.
- Do NOT number the question or add extra commentary.
- Output ONLY the question text itself, nothing else.

Question:"""
    return prompt


test_prompt = build_question_prompt(topics_list[0])
print(test_prompt)

You are an experienced computer science exam paper setter.

Topic: Introduction to Problem Solving: Problem solving steps, algorithms, pseudo-code, flowcharts, overview of Von-Neumann Architecture

Write exactly ONE exam question based on this topic, following these strict rules:
- The question must be worth 5 marks.
- The question should be medium length: not a one-liner, and not a multi-part essay question. Aim for 1-3 sentences.
- Do NOT provide the answer or any explanation.
- Do NOT include the words "Answer:", solutions, or hints.
- Do NOT number the question or add extra commentary.
- Output ONLY the question text itself, nothing else.

Question:


Generate Exam Questions Using DeepSeek

In [24]:
import requests
import re

def generate_question(topic, model="deepseek-r1:8b"):
    """
    Sends the topic prompt to DeepSeek via Ollama and returns
    a clean exam question (with any <think> reasoning removed).
    """
    prompt = build_question_prompt(topic)

    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)
    response.raise_for_status()
    raw_output = response.json()["response"]

    # Remove <think>...</think> reasoning block if present
    cleaned = re.sub(r"<think>.*?</think>", "", raw_output, flags=re.DOTALL)


    cleaned = cleaned.strip()

    return cleaned

# Test with the first topic
question = generate_question(topics_list[0])
print("Generated Question:\n")
print(question)

Generated Question:

Write an algorithm using pseudocode for a simple task and explain how its steps align with the standard problem-solving process while also describing one key aspect of the Von-Neumann architecture relevant to your design.


AI Question Generation

In [ ]:
import time

def generate_all_questions(topics, num_questions=10, model="deepseek-r1:8b"):
    """
    Generates one exam question per topic, up to num_questions.
    Returns a list of question strings.
    """
    selected_topics = topics[:num_questions]  
    questions = []

    for i, topic in enumerate(selected_topics, start=1):
        print(f"Generating question {i} of {num_questions}...")
        question = generate_question(topic, model=model)
        questions.append(question)
        time.sleep(1)  
        
    return questions

# Generate all 10 questions
all_questions = generate_all_questions(topics_list, num_questions=10)

print("\n\n--- All Generated Questions ---\n")
for i, q in enumerate(all_questions, start=1):
    print(f"Q{i}. {q}\n")

Generating question 1 of 10...


Clean and Save Generated exam paper

In [19]:
def clean_question_text(question):
    """
    Removes common instruction-leak phrases that sometimes slip into
    the model's output, and tidies up formatting.
    """
    leak_phrases = [
        r"allocate exactly \d+ marks.*",
        r"worth \d+ marks.*",
        r"\(5 marks\)",
        r"\[5 marks\]",
    ]
    cleaned = question
    for pattern in leak_phrases:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)
    return cleaned.strip()


def save_exam_paper(questions, filename="Generated_Exam.txt"):
    """
    Formats and saves the questions as a numbered exam paper,
    with each question worth 5 marks.
    """
    with open(filename, "w", encoding="utf-8") as f:
        f.write("EXAM PAPER\n")
        f.write("Total Questions: 10 | Marks per Question: 5 | Total Marks: 50\n")
        f.write("=" * 60 + "\n\n")
        for i, q in enumerate(questions, start=1):
            cleaned_q = clean_question_text(q)
            f.write(f"Q{i}. [5 Marks]\n{cleaned_q}\n\n")
    print(f"Exam paper saved as '{filename}'")


# Clean and save
save_exam_paper(all_questions)

Exam paper saved as 'Generated_Exam.txt'
